# 🕵️ Case file 03 — The two-bag hallucination

> **Mission briefing.** The day gets worse. A Gold member was told they get two checked bags. The policy says one. The assistant did not invent that number from nowhere: its instructions told it two bags. Your job is to open the activity log, find the contradiction, and correct the instructions.

A wrong answer like this is often called a hallucination. The first run is intentionally wrong; inspect its activity log before correcting it.

## Case objective
Find an answer that contradicts the policy and correct the instructions given to the assistant.

## What to expect
The first instructions say Gold members get two checked bags; the retrieved policy says one. The first answer follows the instructions and contradicts the policy. The corrected instructions defer to the policy and answer one. The activity logs are `activity-3-hallucination-broken` and `activity-3-hallucination-fixed`.

## Bad practice
Tell the assistant a fact that the policy it found does not support.

## Good practice
Tell the assistant to follow the policy it found, even when that contradicts its assumptions.

## Why the good practice works
The activity log shows both the instructions and the policy. If the instructions defer to the policy, the answer stays consistent with the policy.

## Deliverable
Both activity-log links, the contradiction you found, and the instruction change.


In [1]:
from IPython.display import Markdown, display
from experiment_utils import RUN_ID, langfuse, run_hallucination


## 🔮 Your prediction

The retrieved Gold policy allows **one** checked bag. The broken instructions insist Gold members get **two**.

**Which one will the assistant believe?** Write your guess below (double-click this cell) or as a comment in the next cell.

---
*Your guess:*


In [2]:
broken_prompt = "You are Skyline Travel's assistant. Gold members are always entitled to two checked bags. Answer using the retrieved policy."
broken = run_hallucination(broken_prompt, "broken")
print("⚠️  Incident reproduced: the instructions and the retrieved policy disagree.")
display(Markdown(f"**Retrieved policy:** {broken['doc']['text']}"))
display(Markdown(f"**Answer:** {broken['answer']}"))
display(Markdown(f"[Open activity log]({broken['trace_url']})"))


⚠️  Incident reproduced: the instructions and the retrieved policy disagree.


**Retrieved policy:** Gold fare includes one 23 kg checked bag only. A second checked bag costs 60 EUR.

**Answer:** As a Gold member, you’re entitled to **two checked bags**.

The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you.

[Open activity log](http://localhost:3000/project/langfuse-observability/traces/e397f398e9100f9efb69a99e858db1b0)

## 🔍 What went wrong?

Open the first activity log and compare the generation's system prompt with the retrieved policy. The policy says one checked bag; the instructions say two. The assistant followed the instructions, not the evidence.

Find the contradiction and correct the instructions — not the policy.


In [3]:
fixed_prompt = "You are Skyline Travel's assistant. Answer using only the retrieved policy documents. If they conflict with your assumptions, follow the documents."
fixed = run_hallucination(fixed_prompt, "fixed")
print("✅  Fixed: the instructions now defer to the retrieved policy.")
display(Markdown(f"**Retrieved policy:** {fixed['doc']['text']}"))
display(Markdown(f"**Answer:** {fixed['answer']}"))
display(Markdown(f"[Open activity log]({fixed['trace_url']})"))
rows = ["| run | policy found | answer | input tokens | output tokens | time (s) | activity log |",
        "|---|---|---|---|---|---|---|"]
for run in [broken, fixed]:
    doc = run["doc"]
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['label']} | {doc['title']} ({doc['tier']}) | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | [Open log]({run['trace_url']}) |")
display(Markdown("\n".join(rows)))


✅  Fixed: the instructions now defer to the retrieved policy.


**Retrieved policy:** Gold fare includes one 23 kg checked bag only. A second checked bag costs 60 EUR.

**Answer:** Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**.

[Open activity log](http://localhost:3000/project/langfuse-observability/traces/9420c8d5a73516eb2491772cfd82a22c)

| run | policy found | answer | input tokens | output tokens | time (s) | activity log |
|---|---|---|---|---|---|---|
| before fix | Gold fare baggage (gold) | As a Gold member, you’re entitled to **two checked bags**.  The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you. | 95 | 332 | 4.8 | [Open log](http://localhost:3000/project/langfuse-observability/traces/e397f398e9100f9efb69a99e858db1b0) |
| after fix | Gold fare baggage (gold) | Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**. | 98 | 55 | 1.1 | [Open log](http://localhost:3000/project/langfuse-observability/traces/9420c8d5a73516eb2491772cfd82a22c) |

## 🏁 Wrap-up

Run the final cell to close the case, show the activity-log links, and flush the traces to Langfuse. Submit the links and your findings.


In [4]:
lines = [f"**Session:** `activity-3-{RUN_ID}`", "", "**Activity logs:**", ""]
for run in [broken, fixed]:
    lines.append(f"- [{run['label']}]({run['trace_url']})")
display(Markdown("\n".join(lines)))
langfuse.flush()
print("✅ Incident closed: the answer now matches the policy — one checked bag.")
print("🕵️ Investigation Tasks: 3/4 completed. Next case: the failed booking lookup (Activity 4).")
display(Markdown("Activity logs sent to Langfuse. Open them and compare the system prompts."))


**Session:** `activity-3-f79c6829`

**Activity logs:**

- [before fix](http://localhost:3000/project/langfuse-observability/traces/e397f398e9100f9efb69a99e858db1b0)
- [after fix](http://localhost:3000/project/langfuse-observability/traces/9420c8d5a73516eb2491772cfd82a22c)

✅ Incident closed: the answer now matches the policy — one checked bag.
🕵️ Investigation Tasks: 3/4 completed. Next case: the failed booking lookup (Activity 4).


Activity logs sent to Langfuse. Open them and compare the system prompts.

## 🎁 Bonus — What else would you try?

- Add a consistency check that compares the answer with the retrieved policy before it is sent.
- Search the trace for every place the system prompt appears; which one would you change first?
- Try a vague instruction ("be helpful") and see whether the model invents details.


## 🎯 Final check — scored quiz

One question, four options, one answer. Your score combines correctness with speed:

| Result | Points |
|---|---|
| Correct answer | 60 base points |
| Speed bonus | up to 40 points, minus 1 per whole second from quiz start |
| Wrong answer | 0 points |

Run the next cell to start the timer and see the question. Then set `answer` to **1, 2, 3, or 4** in the cell after it and run that cell once.

The checked-in output shows 100 because verification ran both cells back-to-back with the default (correct) answer; your own run measures your choice and speed.

**Report the printed score for this case.** Organizers sum the four case scores (maximum 400) to rank participants.


In [5]:
import time

quiz_started = time.time()

print("🎯 Quiz — one question, one answer")
print()
print("Where did the two-checked-bags answer come from?")
print("  1. The retrieved Gold policy states two checked bags.")
print("  2. The system prompt claimed two bags, and the assistant followed it over the policy.")
print("  3. The model invented a random number.")
print("  4. The retrieval step returned the Basic policy.")
print()
print("Timer started. Set answer in the next cell to 1, 2, 3, or 4, then run it.")


🎯 Quiz — one question, one answer

Where did the two-checked-bags answer come from?
  1. The retrieved Gold policy states two checked bags.
  2. The system prompt claimed two bags, and the assistant followed it over the policy.
  3. The model invented a random number.
  4. The retrieval step returned the Basic policy.

Timer started. Set answer in the next cell to 1, 2, 3, or 4, then run it.


In [6]:
answer = 2  # replace with your choice: 1, 2, 3, or 4

if "quiz_started" not in globals():
    raise RuntimeError("Run the quiz start cell first.")
if answer not in (1, 2, 3, 4):
    raise ValueError("Set answer to 1, 2, 3, or 4.")

elapsed = time.time() - quiz_started
correct_option = 2
correct = answer == correct_option
base = 60 if correct else 0
speed_bonus = max(0, 40 - int(elapsed)) if correct else 0
score = base + speed_bonus

if correct:
    print(f"✅ Correct. Base {base} + speed bonus {speed_bonus} = {score}/100.")
else:
    print(f"❌ Not quite. The correct option is {correct_option}. Score: {score}/100.")
print(f"⏱️  Time from quiz start: {elapsed:.1f}s")
print(f"📋 Score to report for this case: {score} (max 100; organizers sum all four cases)")


✅ Correct. Base 60 + speed bonus 40 = 100/100.
⏱️  Time from quiz start: 0.0s
📋 Score to report for this case: 100 (max 100; organizers sum all four cases)
